# Deep Learning vs Shallow ML — Oil Recovery Factor Prediction
## MLP · CNN-1D · LSTM · Transformer  vs  RF · XGBoost · SVR · Gradient Boosting

**Dataset:** Proxy5 — polymer flood reservoir simulation  
**DL models:** train / validation / test split (70 / 15 / 15 %)  
**Shallow ML models:** 10-fold cross-validation on the **full dataset**  
**Statistical assessment:** RMSE, MAE, R², MAPE reported as mean ± std across all 10 folds

---
## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
import keras
from keras import layers, Model, Input
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print(f'XGBoost : {xgb.__version__}')
except ImportError:
    print('XGBoost not installed — run: pip install xgboost')
    XGB_AVAILABLE = False

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow : {tf.__version__}')
print(f'Keras      : {keras.__version__}')
print(f'GPU        : {bool(tf.config.list_physical_devices("GPU"))}')

# ── Global Times New Roman font for all figures ───────────────────────────────
plt.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'DejaVu Serif'],
    'axes.titlesize':    12,
    'axes.labelsize':    10,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
    'figure.titlesize':  14,
})

# ── Hyperparameters ───────────────────────────────────────────────────────────
BATCH_SIZE = 64
EPOCHS     = 200
LR         = 1e-3
PATIENCE   = 20
TEST_SIZE  = 0.15
VAL_SIZE   = 0.15
N_FOLDS    = 10

COLOR_MAP = {
    'MLP':               '#2196F3',
    'CNN-1D':            '#FF9800',
    'LSTM':              '#4CAF50',
    'Transformer':       '#9C27B0',
    'Random Forest':     '#F44336',
    'XGBoost':           '#795548',
    'SVR':               '#00BCD4',
    'Gradient Boosting': '#FF5722',
}

DL_MODELS  = ['MLP', 'CNN-1D', 'LSTM', 'Transformer']
ML_MODELS  = ['Random Forest', 'XGBoost', 'SVR', 'Gradient Boosting']
ALL_MODELS = DL_MODELS + ML_MODELS


def draw_architecture(blocks, title, color, filename):
    """Pure-matplotlib block diagram."""
    n = len(blocks)
    fig, ax = plt.subplots(figsize=(6, max(5, n * 0.85 + 1.2)))
    ax.set_xlim(0, 10); ax.set_ylim(0, n + 1); ax.axis('off')
    box_w, box_h, x0 = 7.0, 0.60, 1.5
    for i, (label, shape) in enumerate(blocks):
        y = n - i
        ax.add_patch(mpatches.FancyBboxPatch(
            (x0, y - box_h/2), box_w, box_h, boxstyle='round,pad=0.05',
            facecolor=color, edgecolor='white', alpha=0.85, linewidth=1.5))
        ax.text(5, y,        label, ha='center', va='center',
                fontsize=9, fontweight='bold', color='white',
                fontfamily='serif')
        ax.text(5, y - 0.26, shape, ha='center', va='center',
                fontsize=7, color='white', alpha=0.9, fontfamily='serif')
        if i < n - 1:
            y_next = n - (i + 1)
            ax.annotate('', xy=(5, y_next + box_h/2 + 0.04),
                        xytext=(5, y - box_h/2 - 0.04),
                        arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
    ax.set_title(title, fontsize=12, fontweight='bold', pad=8, fontfamily='serif')
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()

---
## 2. Data Loading & EDA

In [ ]:
df = pd.read_csv('Proxy5.csv', encoding='latin_1').dropna()
print(f'Shape: {df.shape}')
TARGET   = 'Oil_recovery_factor (%)'
FEATURES = [c for c in df.columns if c != TARGET]
print(f'Features ({len(FEATURES)}):', FEATURES)
df.head()

In [ ]:
df.describe().T.style.background_gradient(cmap='YlGnBu', axis=1)

In [ ]:
# ── Fig 1: Target distribution ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(df[TARGET], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Histogram'); axes[0].set_xlabel(TARGET)
axes[1].boxplot(df[TARGET], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='navy', linewidth=2))
axes[1].set_title('Box Plot'); axes[1].set_xticklabels(['Oil Recovery Factor'])
sv = np.sort(df[TARGET])
axes[2].plot(sv, np.arange(1, len(sv)+1)/len(sv), lw=2, color='steelblue')
axes[2].set_title('CDF'); axes[2].set_xlabel(TARGET); axes[2].grid(alpha=0.3)
plt.suptitle('Target Variable Distribution', fontweight='bold')
plt.tight_layout()
plt.savefig('fig01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 2: Feature distributions ─────────────────────────────────────────────
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.ravel()
for i, f in enumerate(FEATURES):
    axes[i].hist(df[f], bins=40, color='#5C85D6', edgecolor='white', alpha=0.85)
    axes[i].set_title(f, fontweight='bold', fontsize=8)
for j in range(len(FEATURES), len(axes)): axes[j].set_visible(False)
plt.suptitle('Feature Distributions', fontweight='bold')
plt.tight_layout()
plt.savefig('fig02_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 3: Correlation heatmap ────────────────────────────────────────────────
plt.figure(figsize=(15, 11))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.4, annot_kws={'size': 7})
plt.title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('fig03_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 4: Features vs Target ─────────────────────────────────────────────────
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.ravel()
for i, f in enumerate(FEATURES):
    axes[i].scatter(df[f], df[TARGET], s=6, alpha=0.15, color='#E05C5C')
    axes[i].set_xlabel(f, fontsize=7); axes[i].set_ylabel('Recovery (%)', fontsize=7)
    r = np.corrcoef(df[f], df[TARGET])[0, 1]
    axes[i].set_title(f'r = {r:.3f}', fontweight='bold', fontsize=8)
for j in range(len(FEATURES), len(axes)): axes[j].set_visible(False)
plt.suptitle('Features vs Oil Recovery Factor', fontweight='bold')
plt.tight_layout()
plt.savefig('fig04_feature_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Pre-processing & Splits

| | DL models | Shallow ML models |
|--|--|--|
| **Strategy** | Train / Val / Test (70/15/15 %) | 10-fold CV on **full dataset** |
| **Train set** | 70 % — gradient updates | 9 folds (90 %) per iteration |
| **Val / CV fold** | 15 % — early stopping | 1 fold (10 %) per iteration |
| **Metrics** | Single train & test evaluation | Mean ± std across 10 folds |

**Scaler:** fitted on each fold's train portion only — no data leakage.

In [ ]:
X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)

# ── DL split: 70% train | 15% val | 15% test ─────────────────────────────────
X_temp, X_test_dl, y_temp, y_test_dl = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED)
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
X_train_dl, X_val_dl, y_train_dl, y_val_dl = train_test_split(
    X_temp, y_temp, test_size=val_ratio, random_state=SEED)

print(f'DL  — Train: {X_train_dl.shape[0]}  Val: {X_val_dl.shape[0]}  Test: {X_test_dl.shape[0]}')
print(f'ML  — Full dataset for 10-fold CV: {X.shape[0]}')

# DL scalers (fit on DL train only)
x_scaler_dl = StandardScaler().fit(X_train_dl)
y_scaler_dl = StandardScaler().fit(y_train_dl.reshape(-1, 1))

X_train_s  = x_scaler_dl.transform(X_train_dl)
X_val_s    = x_scaler_dl.transform(X_val_dl)
X_test_s   = x_scaler_dl.transform(X_test_dl)
y_train_s  = y_scaler_dl.transform(y_train_dl.reshape(-1, 1))
y_val_s    = y_scaler_dl.transform(y_val_dl.reshape(-1, 1))
y_test_s   = y_scaler_dl.transform(y_test_dl.reshape(-1, 1))

N_FEATURES  = X_train_s.shape[1]
X_train_seq = X_train_s.reshape(-1, 1, N_FEATURES)
X_val_seq   = X_val_s.reshape(-1, 1, N_FEATURES)
X_test_seq  = X_test_s.reshape(-1, 1, N_FEATURES)

print(f'N_FEATURES: {N_FEATURES}')

In [ ]:
# ── Fig 5: Dataset split diagram ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].pie(
    [X_train_dl.shape[0], X_val_dl.shape[0], X_test_dl.shape[0]],
    labels=[f'Train ({X_train_dl.shape[0]})', f'Val ({X_val_dl.shape[0]})',
            f'Test ({X_test_dl.shape[0]})'],
    colors=['#4CAF50','#FF9800','#F44336'],
    autopct='%1.1f%%', startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('DL Split  (70 / 15 / 15 %)', fontweight='bold')

# Illustrate one CV fold
fold_sizes = [int(len(X)*0.9), int(len(X)*0.1)]
axes[1].pie(
    fold_sizes,
    labels=[f'Train folds (90%)\n≈{fold_sizes[0]} samples',
            f'Val fold (10%)\n≈{fold_sizes[1]} samples'],
    colors=['#2196F3','#F44336'],
    autopct='%1.1f%%', startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title(f'ML — 10-Fold CV (one fold shown, n={len(X)})', fontweight='bold')

plt.suptitle(f'Dataset Splits  (total n = {len(df)})', fontweight='bold')
plt.tight_layout()
plt.savefig('fig05_dataset_split.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Metric & Training Utilities

In [ ]:
def compute_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).ravel(), np.asarray(y_pred).ravel()
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) /
                          np.where(np.abs(y_true) < 1e-8, 1e-8, y_true))) * 100
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape}


# ── DL utilities ──────────────────────────────────────────────────────────────
def fit_dl(model, name, X_tr, y_tr, X_va, y_va):
    """Train for a fixed EPOCHS so all DL models run the same number of epochs."""
    model.compile(optimizer=keras.optimizers.Adam(LR), loss='mse', metrics=['mae'])
    # ReduceLROnPlateau only — no EarlyStopping, so every model runs for EPOCHS
    callbacks = [ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                   patience=10, min_lr=1e-6, verbose=0)]
    history = model.fit(X_tr, y_tr, validation_data=(X_va, y_va),
                        epochs=EPOCHS, batch_size=BATCH_SIZE,
                        callbacks=callbacks, verbose=0)
    print(f'[{name}] {EPOCHS} epochs — '
          f'train_loss={history.history["loss"][-1]:.5f}  '
          f'val_loss={history.history["val_loss"][-1]:.5f}')
    return history


def eval_dl(model, X_tr, y_tr_s, X_va, y_va_s, X_te, y_te_s, name):
    """Return train, validation & test metrics (original scale) for a Keras model."""
    def inv_pred(X):
        return y_scaler_dl.inverse_transform(model.predict(X, verbose=0)).ravel()
    tr_true = y_scaler_dl.inverse_transform(y_tr_s).ravel()
    va_true = y_scaler_dl.inverse_transform(y_va_s).ravel()
    te_true = y_scaler_dl.inverse_transform(y_te_s).ravel()
    train_m = compute_metrics(tr_true, inv_pred(X_tr))
    val_m   = compute_metrics(va_true, inv_pred(X_va))
    test_m  = compute_metrics(te_true, inv_pred(X_te))
    print(f'[{name}]')
    print(f'  Train RMSE={train_m["RMSE"]:.4f}  R²={train_m["R2"]:.4f}  '
          f'MAE={train_m["MAE"]:.4f}  MAPE={train_m["MAPE"]:.2f}%')
    print(f'  Val   RMSE={val_m["RMSE"]:.4f}  R²={val_m["R2"]:.4f}  '
          f'MAE={val_m["MAE"]:.4f}  MAPE={val_m["MAPE"]:.2f}%')
    print(f'  Test  RMSE={test_m["RMSE"]:.4f}  R²={test_m["R2"]:.4f}  '
          f'MAE={test_m["MAE"]:.4f}  MAPE={test_m["MAPE"]:.2f}%')
    return inv_pred(X_te), te_true, train_m, val_m, test_m


# ── 10-fold CV for shallow ML (full dataset) ──────────────────────────────────
def run_kfold_full(estimator, X_all, y_all, name, n_folds=10):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    oof_preds = np.zeros(len(y_all))
    oof_true  = np.zeros(len(y_all))
    fold_train = {k: [] for k in ['RMSE','MAE','R2','MAPE']}
    fold_val   = {k: [] for k in ['RMSE','MAE','R2','MAPE']}

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all), 1):
        X_tr_f, X_va_f = X_all[tr_idx], X_all[va_idx]
        y_tr_f, y_va_f = y_all[tr_idx], y_all[va_idx]
        scaler_f  = StandardScaler().fit(X_tr_f)
        X_tr_fs   = scaler_f.transform(X_tr_f)
        X_va_fs   = scaler_f.transform(X_va_f)
        estimator.fit(X_tr_fs, y_tr_f)
        oof_preds[va_idx] = estimator.predict(X_va_fs)
        oof_true[va_idx]  = y_va_f
        for k, v in compute_metrics(y_tr_f, estimator.predict(X_tr_fs)).items():
            fold_train[k].append(v)
        for k, v in compute_metrics(y_va_f, estimator.predict(X_va_fs)).items():
            fold_val[k].append(v)

    cv_summary = {}
    print(f'\n[{name}]  10-Fold CV (full dataset, n={len(y_all)})')
    print(f'  {"Metric":<8}  {"Train fold mean±std":>24}  {"Val fold mean±std":>24}')
    print('  ' + '-'*56)
    for k in ['RMSE','MAE','R2','MAPE']:
        tr_arr = np.array(fold_train[k])
        va_arr = np.array(fold_val[k])
        label  = f'{k}(%)' if k == 'MAPE' else k
        print(f'  {label:<8}  {tr_arr.mean():>10.4f} ± {tr_arr.std():.4f}  '
              f'  {va_arr.mean():>10.4f} ± {va_arr.std():.4f}')
        cv_summary[f'train_{k}_mean'] = tr_arr.mean()
        cv_summary[f'train_{k}_std']  = tr_arr.std()
        cv_summary[f'val_{k}_mean']   = va_arr.mean()
        cv_summary[f'val_{k}_std']    = va_arr.std()

    oof_m = compute_metrics(oof_true, oof_preds)
    print(f'  OOF RMSE={oof_m["RMSE"]:.4f}  MAE={oof_m["MAE"]:.4f}  '
          f'R²={oof_m["R2"]:.4f}  MAPE={oof_m["MAPE"]:.2f}%')
    return oof_preds, oof_true, oof_m, cv_summary, fold_val

---
## 5. DL Model 1 — MLP

In [ ]:
def build_mlp(n_features, dropout=0.2):
    inp = Input(shape=(n_features,))
    x = layers.Dense(128, activation='relu')(inp)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(64,  activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32,  activation='relu')(x)
    return Model(inp, layers.Dense(1)(x), name='MLP')

mlp_model = build_mlp(N_FEATURES)
mlp_model.summary()

In [ ]:
draw_architecture([
    ('Input',                           f'(None, {N_FEATURES})'),
    ('Dense 128 + ReLU + Dropout(0.2)', '(None, 128)'),
    ('Dense  64 + ReLU + Dropout(0.2)', '(None,  64)'),
    ('Dense  32 + ReLU',                '(None,  32)'),
    ('Dense   1  (Output)',             '(None,   1)'),
], title='MLP Architecture', color=COLOR_MAP['MLP'], filename='fig06a_mlp_arch.png')

In [ ]:
mlp_hist = fit_dl(mlp_model, 'MLP', X_train_s, y_train_s, X_val_s, y_val_s)
mlp_preds, mlp_trues, mlp_train_m, mlp_val_m, mlp_test_m = eval_dl(
    mlp_model, X_train_s, y_train_s, X_val_s, y_val_s, X_test_s, y_test_s, 'MLP')

---
## 6. DL Model 2 — CNN-1D

In [ ]:
def build_cnn(n_features, dropout=0.2):
    inp = Input(shape=(1, n_features))
    x = layers.ZeroPadding1D(1)(inp)
    x = layers.Conv1D(64, 3, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.ZeroPadding1D(1)(x)
    x = layers.Conv1D(32, 3, activation='relu')(x)
    x = layers.Flatten()(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    return Model(inp, layers.Dense(1)(x), name='CNN_1D')

cnn_model = build_cnn(N_FEATURES)
cnn_model.summary()

In [ ]:
draw_architecture([
    ('Input',                       f'(None, 1, {N_FEATURES})'),
    ('ZeroPad1D(1)',                 f'(None, 3, {N_FEATURES})'),
    ('Conv1D 64 k=3 + ReLU + Drop',  '(None, 1, 64)'),
    ('ZeroPad1D(1)',                  '(None, 3, 64)'),
    ('Conv1D 32 k=3 + ReLU',         '(None, 1, 32)'),
    ('Flatten',                      '(None, 32)'),
    ('Dense 32 + ReLU + Dropout',    '(None, 32)'),
    ('Dense  1 (Output)',            '(None,  1)'),
], title='CNN-1D Architecture', color=COLOR_MAP['CNN-1D'], filename='fig06b_cnn_arch.png')

In [ ]:
cnn_hist = fit_dl(cnn_model, 'CNN-1D', X_train_seq, y_train_s, X_val_seq, y_val_s)
cnn_preds, cnn_trues, cnn_train_m, cnn_val_m, cnn_test_m = eval_dl(
    cnn_model, X_train_seq, y_train_s, X_val_seq, y_val_s, X_test_seq, y_test_s, 'CNN-1D')

---
## 7. DL Model 3 — LSTM

In [ ]:
def build_lstm(n_features, hidden=64, dropout=0.2):
    inp = Input(shape=(1, n_features))
    x = layers.LSTM(hidden, dropout=dropout)(inp)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    return Model(inp, layers.Dense(1)(x), name='LSTM')

lstm_model = build_lstm(N_FEATURES)
lstm_model.summary()

In [ ]:
draw_architecture([
    ('Input',                    f'(None, 1, {N_FEATURES})'),
    ('LSTM 64 + Dropout',         '(None, 64)'),
    ('Dense 32 + ReLU + Dropout', '(None, 32)'),
    ('Dense  1 (Output)',         '(None,  1)'),
], title='LSTM Architecture', color=COLOR_MAP['LSTM'], filename='fig06c_lstm_arch.png')

In [ ]:
lstm_hist = fit_dl(lstm_model, 'LSTM', X_train_seq, y_train_s, X_val_seq, y_val_s)
lstm_preds, lstm_trues, lstm_train_m, lstm_val_m, lstm_test_m = eval_dl(
    lstm_model, X_train_seq, y_train_s, X_val_seq, y_val_s, X_test_seq, y_test_s, 'LSTM')

---
## 8. DL Model 4 — Transformer

In [ ]:
def build_transformer(n_features, num_heads=2, key_dim=64, dropout=0.1):
    inp  = Input(shape=(1, n_features))
    attn = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=key_dim, dropout=dropout)(inp, inp)
    attn = layers.LayerNormalization()(attn + inp)
    x    = layers.GlobalAveragePooling1D()(attn)
    x    = layers.Dense(64, activation='relu')(x)
    x    = layers.Dropout(dropout)(x)
    x    = layers.Dense(32, activation='relu')(x)
    return Model(inp, layers.Dense(1)(x), name='Transformer')

tfm_model = build_transformer(N_FEATURES)
tfm_model.summary()

In [ ]:
draw_architecture([
    ('Input',                       f'(None, 1, {N_FEATURES})'),
    ('MultiHeadAttention (2 heads)', '(None, 1, N_FEATURES)'),
    ('Add & LayerNorm',              '(None, 1, N_FEATURES)'),
    ('GlobalAveragePooling1D',       '(None, N_FEATURES)'),
    ('Dense 64 + ReLU + Dropout',    '(None, 64)'),
    ('Dense 32 + ReLU',              '(None, 32)'),
    ('Dense  1 (Output)',            '(None,  1)'),
], title='Transformer Architecture', color=COLOR_MAP['Transformer'], filename='fig06d_tfm_arch.png')

In [ ]:
tfm_hist = fit_dl(tfm_model, 'Transformer', X_train_seq, y_train_s, X_val_seq, y_val_s)
tfm_preds, tfm_trues, tfm_train_m, tfm_val_m, tfm_test_m = eval_dl(
    tfm_model, X_train_seq, y_train_s, X_val_seq, y_val_s, X_test_seq, y_test_s, 'Transformer')

---
## 9. Shallow ML — 10-Fold CV on Full Dataset

In [ ]:
rf_est = RandomForestRegressor(n_estimators=300, min_samples_leaf=2,
                                n_jobs=-1, random_state=SEED)
rf_preds, rf_trues, rf_oof_m, rf_cv, rf_folds = run_kfold_full(
    rf_est, X, y, 'Random Forest')

In [ ]:
if XGB_AVAILABLE:
    xgb_est = xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=SEED, n_jobs=-1, verbosity=0)
    xgb_preds, xgb_trues, xgb_oof_m, xgb_cv, xgb_folds = run_kfold_full(
        xgb_est, X, y, 'XGBoost')
else:
    print('XGBoost not installed — skipped.')
    xgb_preds  = np.full(len(y), np.nan)
    xgb_trues  = y.copy()
    xgb_oof_m  = {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan, 'MAPE': np.nan}
    xgb_cv     = {k: np.nan for k in ['train_RMSE_mean','train_R2_mean',
                                        'val_RMSE_mean','val_R2_mean']}
    xgb_folds  = {k: [np.nan]*N_FOLDS for k in ['RMSE','MAE','R2','MAPE']}

In [ ]:
svr_est = SVR(kernel='rbf', C=10, epsilon=0.01, gamma='scale')
svr_preds, svr_trues, svr_oof_m, svr_cv, svr_folds = run_kfold_full(
    svr_est, X, y, 'SVR')

In [ ]:
gbm_est = GradientBoostingRegressor(
    n_estimators=400, learning_rate=0.05, max_depth=5,
    subsample=0.8, min_samples_leaf=3, random_state=SEED)
gbm_preds, gbm_trues, gbm_oof_m, gbm_cv, gbm_folds = run_kfold_full(
    gbm_est, X, y, 'Gradient Boosting')

---
## 10. Combined Results

In [ ]:
all_preds = {
    'MLP':               (mlp_trues,  mlp_preds),
    'CNN-1D':            (cnn_trues,  cnn_preds),
    'LSTM':              (lstm_trues, lstm_preds),
    'Transformer':       (tfm_trues,  tfm_preds),
    'Random Forest':     (rf_trues,   rf_preds),
    'XGBoost':           (xgb_trues,  xgb_preds),
    'SVR':               (svr_trues,  svr_preds),
    'Gradient Boosting': (gbm_trues,  gbm_preds),
}

primary_m = {
    'MLP': mlp_test_m,   'CNN-1D': cnn_test_m,
    'LSTM': lstm_test_m, 'Transformer': tfm_test_m,
    'Random Forest': rf_oof_m,   'XGBoost': xgb_oof_m,
    'SVR': svr_oof_m,    'Gradient Boosting': gbm_oof_m,
}
train_m_dl = {
    'MLP': mlp_train_m, 'CNN-1D': cnn_train_m,
    'LSTM': lstm_train_m, 'Transformer': tfm_train_m,
}
val_m_dl = {
    'MLP': mlp_val_m,   'CNN-1D': cnn_val_m,
    'LSTM': lstm_val_m, 'Transformer': tfm_val_m,
}
cv_map = {
    'Random Forest': rf_cv,  'XGBoost': xgb_cv,
    'SVR': svr_cv,           'Gradient Boosting': gbm_cv,
}

rows = []
for name in ALL_MODELS:
    pm  = primary_m[name]
    row = {'Model': name,
           'Type': 'Deep Learning' if name in DL_MODELS else 'Shallow ML',
           'Eval':  'Test set'     if name in DL_MODELS else 'OOF (10-fold)',
           'RMSE': pm['RMSE'], 'MAE': pm['MAE'],
           'R²':   pm['R2'],   'MAPE(%)': pm['MAPE']}
    if name in DL_MODELS:
        row['Train R²']    = train_m_dl[name]['R2']
        row['Val R²']      = val_m_dl[name]['R2']
        row['Train RMSE']  = train_m_dl[name]['RMSE']
        row['Val RMSE']    = val_m_dl[name]['RMSE']
        row['Train MAE']   = train_m_dl[name]['MAE']
        row['Val MAE']     = val_m_dl[name]['MAE']
    else:
        cv = cv_map[name]
        row['Train R²']   = cv.get('train_R2_mean',   np.nan)
        row['Val R²']     = cv.get('val_R2_mean',     np.nan)
        row['Train RMSE'] = cv.get('train_RMSE_mean', np.nan)
        row['Val RMSE']   = cv.get('val_RMSE_mean',   np.nan)
        row['Train MAE']  = cv.get('train_MAE_mean',  np.nan)
        row['Val MAE']    = cv.get('val_MAE_mean',    np.nan)
    rows.append(row)

results = pd.DataFrame(rows)   # keep DL_MODELS + ML_MODELS order (no sort)

print('\n' + '='*100)
print('  RESULTS  (DL = test-set | ML = OOF from 10-fold CV on full dataset)')
print('='*100)
cols_show = ['Model','Type','Eval','Train R²','Val R²','R²',
             'Train RMSE','Val RMSE','RMSE','Train MAE','Val MAE','MAE','MAPE(%)']
print(results[cols_show].to_string(index=False))

results[cols_show].style \
    .background_gradient(subset=['RMSE','MAE','MAPE(%)'], cmap='RdYlGn_r') \
    .background_gradient(subset=['R²'], cmap='RdYlGn') \
    .format({c: '{:.4f}' for c in cols_show
             if c not in ['Model','Type','Eval']})

In [ ]:
# ── Statistical assessment table — ML models ─────────────────────────────────
folds_map = {'Random Forest': rf_folds, 'XGBoost': xgb_folds,
             'SVR': svr_folds, 'Gradient Boosting': gbm_folds}

stat_rows = []
for name in ML_MODELS:
    cv = cv_map[name]
    stat_rows.append({
        'Model':                    name,
        'CV Train R²   (mean±std)': f'{cv.get("train_R2_mean",np.nan):.4f} ± {cv.get("train_R2_std",np.nan):.4f}',
        'CV Val  R²   (mean±std)':  f'{cv.get("val_R2_mean",np.nan):.4f} ± {cv.get("val_R2_std",np.nan):.4f}',
        'CV Train RMSE (mean±std)': f'{cv.get("train_RMSE_mean",np.nan):.4f} ± {cv.get("train_RMSE_std",np.nan):.4f}',
        'CV Val  RMSE (mean±std)':  f'{cv.get("val_RMSE_mean",np.nan):.4f} ± {cv.get("val_RMSE_std",np.nan):.4f}',
        'CV Train MAE  (mean±std)': f'{cv.get("train_MAE_mean",np.nan):.4f} ± {cv.get("train_MAE_std",np.nan):.4f}',
        'CV Val  MAE  (mean±std)':  f'{cv.get("val_MAE_mean",np.nan):.4f} ± {cv.get("val_MAE_std",np.nan):.4f}',
        'CV Train MAPE%(mean±std)': f'{cv.get("train_MAPE_mean",np.nan):.4f} ± {cv.get("train_MAPE_std",np.nan):.4f}',
        'CV Val  MAPE%(mean±std)':  f'{cv.get("val_MAPE_mean",np.nan):.4f} ± {cv.get("val_MAPE_std",np.nan):.4f}',
        'OOF R²':   f'{primary_m[name]["R2"]:.4f}',
        'OOF RMSE': f'{primary_m[name]["RMSE"]:.4f}',
        'OOF MAE':  f'{primary_m[name]["MAE"]:.4f}',
        'OOF MAPE%':f'{primary_m[name]["MAPE"]:.4f}',
    })

stat_df = pd.DataFrame(stat_rows)
print('\n10-Fold CV Statistical Assessment — Shallow ML Models (full dataset)')
print('='*110)
print(stat_df.to_string(index=False))

---
## 11. Figures

### 11.1 DL Learning Curves (Fig 7)

In [ ]:
dl_hists = {'MLP': mlp_hist, 'CNN-1D': cnn_hist,
             'LSTM': lstm_hist, 'Transformer': tfm_hist}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, hist in dl_hists.items():
    c = COLOR_MAP[name]
    axes[0].plot(hist.history['loss'],     label=name, color=c, lw=1.8)
    axes[1].plot(hist.history['val_loss'], label=name, color=c, lw=1.8)
for ax, title in zip(axes, ['Training Loss (MSE)', 'Validation Loss (MSE)']):
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE (scaled)'); ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('DL Learning Curves', fontweight='bold')
plt.tight_layout()
plt.savefig('fig07_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.2 Train vs Test/OOF R² — All 8 Models (Fig 8)

In [ ]:
# ── Fig 8: R² grouped bar chart ───────────────────────────────────────────────
# DL: 3 bars (Train / Validation / Test) — ML: 2 bars (CV Train fold / OOF)
# Models kept in fixed order: DL first, then ML

x = np.arange(len(ALL_MODELS))
w = 0.22   # bar width

fig, ax = plt.subplots(figsize=(17, 6))

legend_handles = []
for i, name in enumerate(ALL_MODELS):
    c = COLOR_MAP[name]
    row = results[results['Model'] == name].iloc[0]

    if name in DL_MODELS:
        b1 = ax.bar(x[i] - w,   row['Train R²'], w, color=c, alpha=0.30,
                    edgecolor='white', linewidth=0.8)
        b2 = ax.bar(x[i],       row['Val R²'],   w, color=c, alpha=0.65,
                    edgecolor='white', linewidth=0.8)
        b3 = ax.bar(x[i] + w,   row['R²'],       w, color=c, alpha=1.00,
                    edgecolor='white', linewidth=0.8)
        for bar, v in [(b1, row['Train R²']), (b2, row['Val R²']), (b3, row['R²'])]:
            ax.text(bar[0].get_x() + bar[0].get_width()/2,
                    bar[0].get_height() + 0.004,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=6.5)
        if i == 0:
            legend_handles += [
                mpatches.Patch(facecolor='dimgray', alpha=0.30, label='DL — Train'),
                mpatches.Patch(facecolor='dimgray', alpha=0.65, label='DL — Validation'),
                mpatches.Patch(facecolor='dimgray', alpha=1.00, label='DL — Test'),
            ]
    else:
        b1 = ax.bar(x[i] - w/2, row['Train R²'], w, color=c, alpha=0.30,
                    edgecolor='white', linewidth=0.8)
        b2 = ax.bar(x[i] + w/2, row['R²'],       w, color=c, alpha=1.00,
                    edgecolor='white', linewidth=0.8)
        for bar, v in [(b1, row['Train R²']), (b2, row['R²'])]:
            ax.text(bar[0].get_x() + bar[0].get_width()/2,
                    bar[0].get_height() + 0.004,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=6.5)
        if i == len(DL_MODELS):
            legend_handles += [
                mpatches.Patch(facecolor='dimgray', alpha=0.30, label='ML — CV Train fold'),
                mpatches.Patch(facecolor='dimgray', alpha=1.00, label='ML — OOF'),
            ]

ax.set_xticks(x)
ax.set_xticklabels(ALL_MODELS, rotation=20, ha='right')
ax.set_ylabel('R²  (↑ better)')
ax.set_title('R² by Split Phase — DL (Train / Val / Test)  |  ML (CV Train / OOF)',
             fontweight='bold')
ax.axvline(3.5, color='gray', linestyle='--', lw=1.2)
ax.text(1.5,  0.02, 'Deep Learning', ha='center', color='gray', fontsize=9)
ax.text(5.5,  0.02, 'Shallow ML',    ha='center', color='gray', fontsize=9)
ax.legend(handles=legend_handles, fontsize=9, loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig08_train_test_r2.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.3 Train vs Test/OOF RMSE — All 8 Models (Fig 9)

In [ ]:
# ── Fig 9: RMSE grouped bar chart ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(17, 6))

legend_handles = []
for i, name in enumerate(ALL_MODELS):
    c = COLOR_MAP[name]
    row = results[results['Model'] == name].iloc[0]

    if name in DL_MODELS:
        b1 = ax.bar(x[i] - w,   row['Train RMSE'], w, color=c, alpha=0.30,
                    edgecolor='white', linewidth=0.8)
        b2 = ax.bar(x[i],       row['Val RMSE'],   w, color=c, alpha=0.65,
                    edgecolor='white', linewidth=0.8)
        b3 = ax.bar(x[i] + w,   row['RMSE'],       w, color=c, alpha=1.00,
                    edgecolor='white', linewidth=0.8)
        for bar, v in [(b1, row['Train RMSE']), (b2, row['Val RMSE']), (b3, row['RMSE'])]:
            ax.text(bar[0].get_x() + bar[0].get_width()/2,
                    bar[0].get_height() + 0.002,
                    f'{v:.4f}', ha='center', va='bottom', fontsize=6.5)
        if i == 0:
            legend_handles += [
                mpatches.Patch(facecolor='dimgray', alpha=0.30, label='DL — Train'),
                mpatches.Patch(facecolor='dimgray', alpha=0.65, label='DL — Validation'),
                mpatches.Patch(facecolor='dimgray', alpha=1.00, label='DL — Test'),
            ]
    else:
        b1 = ax.bar(x[i] - w/2, row['Train RMSE'], w, color=c, alpha=0.30,
                    edgecolor='white', linewidth=0.8)
        b2 = ax.bar(x[i] + w/2, row['RMSE'],       w, color=c, alpha=1.00,
                    edgecolor='white', linewidth=0.8)
        for bar, v in [(b1, row['Train RMSE']), (b2, row['RMSE'])]:
            ax.text(bar[0].get_x() + bar[0].get_width()/2,
                    bar[0].get_height() + 0.002,
                    f'{v:.4f}', ha='center', va='bottom', fontsize=6.5)
        if i == len(DL_MODELS):
            legend_handles += [
                mpatches.Patch(facecolor='dimgray', alpha=0.30, label='ML — CV Train fold'),
                mpatches.Patch(facecolor='dimgray', alpha=1.00, label='ML — OOF'),
            ]

ax.set_xticks(x)
ax.set_xticklabels(ALL_MODELS, rotation=20, ha='right')
ax.set_ylabel('RMSE  (↓ better)')
ax.set_title('RMSE by Split Phase — DL (Train / Val / Test)  |  ML (CV Train / OOF)',
             fontweight='bold')
ax.axvline(3.5, color='gray', linestyle='--', lw=1.2)
ax.text(1.5,  ax.get_ylim()[0], 'Deep Learning', ha='center', color='gray', fontsize=9)
ax.text(5.5,  ax.get_ylim()[0], 'Shallow ML',    ha='center', color='gray', fontsize=9)
ax.legend(handles=legend_handles, fontsize=9, loc='upper right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig09_train_test_rmse.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.4 All Metrics — Test/OOF (Fig 10)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
bar_colors = [COLOR_MAP[m] for m in model_order]
for ax, (col, lower) in zip(axes,
        [('RMSE',True),('MAE',True),('R²',False),('MAPE(%)',True)]):
    vals = results[col].tolist()
    bars = ax.bar(model_order, vals, color=bar_colors, edgecolor='white')
    ax.set_title(col, fontweight='bold')
    ax.set_xticklabels(model_order, rotation=30, ha='right')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+max(vals)*0.01,
                f'{v:.4f}', ha='center', va='bottom', fontsize=7)
    ax.set_xlabel('↓ better' if lower else '↑ better', color='gray')
    ax.grid(axis='y', alpha=0.3)
plt.suptitle('Test / OOF Metrics — All 8 Models', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig10_metric_bars.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.5 10-Fold CV Validation Boxplots — Shallow ML (Fig 11)

In [ ]:
ml_colors = [COLOR_MAP[m] for m in ML_MODELS]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, metric in zip(axes, ['R2','RMSE','MAE','MAPE']):
    data = [folds_map[m][metric] for m in ML_MODELS]
    bp   = ax.boxplot(data, labels=ML_MODELS, patch_artist=True,
                      medianprops=dict(color='black', linewidth=2))
    for patch, c in zip(bp['boxes'], ml_colors):
        patch.set_facecolor(c); patch.set_alpha(0.75)
    label = 'R²  (↑)' if metric == 'R2' else f'{metric}  (↓)'
    ax.set_title(label, fontweight='bold')
    ax.set_xticklabels(ML_MODELS, rotation=20, ha='right')
    ax.grid(axis='y', alpha=0.3)
plt.suptitle('10-Fold CV Validation Metrics — Shallow ML (full dataset)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('fig11_cv_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.6 Actual vs Predicted — All 8 Models (Fig 12)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for ax, name in zip(axes.ravel(), model_order):
    t, p = all_preds[name]
    r2   = r2_score(t, p)
    rmse = np.sqrt(mean_squared_error(t, p))
    ax.scatter(t, p, alpha=0.35, s=12, color=COLOR_MAP[name], edgecolors='none')
    lo, hi = min(t.min(), p.min()), max(t.max(), p.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1.5)
    scheme = 'Test set' if name in DL_MODELS else 'OOF'
    ax.set_title(f'{name}  [{scheme}]\nR²={r2:.4f}  RMSE={rmse:.4f}', fontsize=9)
    ax.set_xlabel('Actual (%)'); ax.set_ylabel('Predicted (%)')
    ax.grid(alpha=0.25)
plt.suptitle('Actual vs. Predicted', fontweight='bold')
plt.tight_layout()
plt.savefig('fig12_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.7 Residual Distributions (Fig 13)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 8))
for ax, name in zip(axes.ravel(), model_order):
    t, p = all_preds[name]; res = t - p
    ax.hist(res, bins=50, color=COLOR_MAP[name], edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', linestyle='--', linewidth=1.5)
    ax.set_title(f'{name}\nmean={res.mean():.4f}')
    ax.set_xlabel('Residual'); ax.set_ylabel('Count'); ax.grid(alpha=0.25)
plt.suptitle('Residual Distributions', fontweight='bold')
plt.tight_layout()
plt.savefig('fig13_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.8 Residual vs Predicted (Fig 14)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 8))
for ax, name in zip(axes.ravel(), model_order):
    t, p = all_preds[name]; res = t - p
    ax.scatter(p, res, alpha=0.3, s=10, color=COLOR_MAP[name], edgecolors='none')
    ax.axhline(0, color='black', linestyle='--', linewidth=1.5)
    ax.set_title(name); ax.set_xlabel('Predicted (%)'); ax.set_ylabel('Residual')
    ax.grid(alpha=0.25)
plt.suptitle('Residual vs. Predicted — Homoscedasticity Check', fontweight='bold')
plt.tight_layout()
plt.savefig('fig14_residual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.9 Absolute Error Box Plots (Fig 15)

In [ ]:
abs_errors = [np.abs(all_preds[m][0] - all_preds[m][1]) for m in model_order]
fig, ax = plt.subplots(figsize=(13, 5))
bp = ax.boxplot(abs_errors, labels=model_order, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
for patch, name in zip(bp['boxes'], model_order):
    patch.set_facecolor(COLOR_MAP[name]); patch.set_alpha(0.75)
ax.set_title('Absolute Error Distribution', fontweight='bold')
ax.set_ylabel('|Actual − Predicted| (%)')
ax.set_xticklabels(model_order, rotation=20, ha='right')
ax.axvspan(0.5, 4.5, alpha=0.05, color='blue',  label='Deep Learning')
ax.axvspan(4.5, 8.5, alpha=0.05, color='red',   label='Shallow ML')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig15_abs_error_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.10 Radar Chart (Fig 16)

In [ ]:
metric_cols  = ['RMSE', 'MAE', 'R²', 'MAPE(%)']
radar_labels = ['1-RMSE\n(norm)', '1-MAE\n(norm)', 'R²\n(norm)', '1-MAPE\n(norm)']

vals = results[metric_cols].values.astype(float)
norm = np.zeros_like(vals)
for j, col in enumerate(metric_cols):
    lo, hi = np.nanmin(vals[:, j]), np.nanmax(vals[:, j])
    norm[:, j] = ((vals[:, j] - lo) / (hi - lo + 1e-12) if col == 'R²'
                  else 1 - (vals[:, j] - lo) / (hi - lo + 1e-12))

N = len(metric_cols)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for i, name in enumerate(model_order):
    ri = results.index[results['Model'] == name][0]
    v  = norm[ri].tolist() + [norm[ri][0]]
    ls = '-' if name in DL_MODELS else '--'
    ax.plot(angles, v, lw=2, color=COLOR_MAP[name], linestyle=ls, label=name)
    ax.fill(angles, v, alpha=0.05, color=COLOR_MAP[name])
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels)
ax.set_ylim(0, 1)
ax.set_title('Radar Chart — All Models  (outer = better)',
             fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.45, 1.2))
plt.tight_layout()
plt.savefig('fig16_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.11 Permutation Feature Importance — Best Model (Fig 17)

In [ ]:
def perm_imp_dl(model, X_s, y_true, y_scaler, feature_names, n_repeats=5, is_seq=False):
    def pred(X):
        Xi = X.reshape(-1, 1, X.shape[1]) if is_seq else X
        return y_scaler.inverse_transform(model.predict(Xi, verbose=0)).ravel()
    base = np.sqrt(mean_squared_error(y_true, pred(X_s)))
    imp  = np.zeros((len(feature_names), n_repeats))
    for i in range(len(feature_names)):
        for r in range(n_repeats):
            Xp = X_s.copy(); np.random.shuffle(Xp[:, i])
            imp[i, r] = np.sqrt(mean_squared_error(y_true, pred(Xp))) - base
    return imp.mean(1), imp.std(1)


def perm_imp_sklearn(model, X_s, y_true, feature_names, n_repeats=5):
    base = np.sqrt(mean_squared_error(y_true, model.predict(X_s)))
    imp  = np.zeros((len(feature_names), n_repeats))
    for i in range(len(feature_names)):
        for r in range(n_repeats):
            Xp = X_s.copy(); np.random.shuffle(Xp[:, i])
            imp[i, r] = np.sqrt(mean_squared_error(y_true, model.predict(Xp))) - base
    return imp.mean(1), imp.std(1)


best_name = results.iloc[0]['Model']
print(f'Best model: {best_name}')

dl_map = {'MLP': (mlp_model, False, X_test_s,  y_test_dl),
           'CNN-1D':      (cnn_model,  True,  X_test_seq, y_test_dl),
           'LSTM':        (lstm_model, True,  X_test_seq, y_test_dl),
           'Transformer': (tfm_model,  True,  X_test_seq, y_test_dl)}

# For ML: use a global scaler fitted on all X for permutation
x_scaler_all = StandardScaler().fit(X)
X_all_s      = x_scaler_all.transform(X)

ml_est_map = {'Random Forest': rf_est, 'XGBoost': xgb_est if XGB_AVAILABLE else None,
              'SVR': svr_est, 'Gradient Boosting': gbm_est}

if best_name in dl_map:
    bm, is_seq, X_ev, y_ev = dl_map[best_name]
    imp_mean, imp_std = perm_imp_dl(
        bm, X_ev, y_ev, y_scaler_dl, FEATURES, is_seq=is_seq)
else:
    bm = ml_est_map[best_name]
    # Refit on full scaled data for importance
    bm.fit(X_all_s, y)
    imp_mean, imp_std = perm_imp_sklearn(bm, X_all_s, y, FEATURES)

sorted_idx = np.argsort(imp_mean)[::-1]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].barh([FEATURES[i] for i in sorted_idx],
             imp_mean[sorted_idx], xerr=imp_std[sorted_idx],
             color=COLOR_MAP[best_name], alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Mean RMSE increase')
axes[0].set_title(f'Permutation Importance — {best_name}', fontweight='bold')
axes[0].invert_yaxis(); axes[0].grid(axis='x', alpha=0.3)

imp_df = pd.DataFrame({'Feature': FEATURES, 'Importance': imp_mean}).set_index('Feature')
sns.heatmap(imp_df.sort_values('Importance', ascending=False)[['Importance']].T,
            ax=axes[1], cmap='YlOrRd', annot=True, fmt='.4f', linewidths=0.5,
            cbar_kws={'label': 'RMSE increase'}, annot_kws={'size': 7})
axes[1].set_title('Feature Importance Heatmap', fontweight='bold')
axes[1].set_yticklabels(['Importance'], rotation=0)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.suptitle(f'Feature Importance — {best_name} (Best Model)', fontweight='bold')
plt.tight_layout()
plt.savefig('fig17_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.12 Prediction Error Band — All 8 Models (Fig 18)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for ax, name in zip(axes.ravel(), model_order):
    t, p = all_preds[name]
    idx  = np.argsort(t); tv, pv = t[idx], p[idx]
    xi   = np.arange(len(tv))
    ax.fill_between(xi, tv, pv, alpha=0.35, color=COLOR_MAP[name], label='Error band')
    ax.plot(xi, tv, 'k-', lw=0.8, label='Actual')
    ax.plot(xi, pv, '--', lw=0.8, color=COLOR_MAP[name], label='Predicted')
    ax.set_title(f'{name}  MAE={np.abs(tv-pv).mean():.4f}')
    ax.set_xlabel('Sorted sample index'); ax.set_ylabel('Oil Recovery (%)')
    ax.legend(fontsize=7); ax.grid(alpha=0.25)
plt.suptitle('Prediction Error Band (sorted by actual value)', fontweight='bold')
plt.tight_layout()
plt.savefig('fig18_error_band.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 12. Final Summary

In [ ]:
summary = results[['Model','Type','Eval scheme','Train R²','R²',
                    'Train RMSE','RMSE','Train MAE','MAE','MAPE(%)']].copy()
summary.insert(0, 'Rank', range(1, len(summary)+1))

print('\n' + '='*110)
print('  FINAL RANKING  (DL = test-set | ML = OOF from 10-fold CV on full dataset)')
print('='*110)
print(summary.to_string(index=False))
print()

dl_r2 = results.loc[results['Type']=='Deep Learning', 'R²'].mean()
ml_r2 = results.loc[results['Type']=='Shallow ML',    'R²'].mean()
print(f'Average R²  — Deep Learning : {dl_r2:.4f}')
print(f'Average R²  — Shallow ML    : {ml_r2:.4f}')
winner = 'Deep Learning' if dl_r2 > ml_r2 else 'Shallow ML'
print(f'\n>>> Higher average R²: {winner} <<<')